# [Francois Chollet's Blog](https://blog.keras.io/building-powerful-image-classification-models-using-very-little-data.html)

## Simple architecture

In [ ]:
import os
import h5py
import numpy as np

from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Convolution2D, MaxPooling2D, ZeroPadding2D
from keras.layers import Activation, Dropout, Flatten, Dense
from keras import optimizers

from keras import backend as K
K.set_image_dim_ordering('th')

In [ ]:
# dimensions of our images.
img_width, img_height = 150, 150

folder = 'data/catsdogs/'
train_data_dir = folder+'train'
validation_data_dir = folder+'validation'

weights_path = 'data/vgg16_weights.h5'
top_model_weights_path = folder+'savedmodels/bottleneck_fc_model.h5'

nb_train_samples = 2222
nb_validation_samples = 1222
nb_epoch = 50 # originally 50

Train_Bottleneck = False
nb_class = 2

In [ ]:
simplemodel = Sequential()
simplemodel.add(Convolution2D(32, 3, 3, input_shape=(3, img_width, img_height)))
simplemodel.add(Activation('relu'))
simplemodel.add(MaxPooling2D(pool_size=(2, 2)))

simplemodel.add(Convolution2D(32, 3, 3))
simplemodel.add(Activation('relu'))
simplemodel.add(MaxPooling2D(pool_size=(2, 2)))

simplemodel.add(Convolution2D(64, 3, 3))
simplemodel.add(Activation('relu'))
simplemodel.add(MaxPooling2D(pool_size=(2, 2)))

simplemodel.add(Flatten())
simplemodel.add(Dense(64))
simplemodel.add(Activation('relu'))
simplemodel.add(Dropout(0.5))
simplemodel.add(Dense(1))
simplemodel.add(Activation('sigmoid'))

simplemodel.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [ ]:
# this is the augmentation configuration we will use for training
train_datagen = ImageDataGenerator(
        rescale=1./255,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True)

# this is the augmentation configuration we will use for testing:
# only rescaling
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
        train_data_dir,
        target_size=(img_width, img_height),
        batch_size=32,
        class_mode='binary')

validation_generator = test_datagen.flow_from_directory(
        validation_data_dir,
        target_size=(img_width, img_height),
        batch_size=32,
        class_mode='binary')

In [ ]:
print simplemodel.summary()
simplemodel.fit_generator(
        train_generator,
        samples_per_epoch=nb_train_samples,
        nb_epoch=nb_epoch,
        validation_data=validation_generator,
        nb_val_samples=nb_validation_samples)

In [ ]:
simplemodel.save_weights(folder+'savedmodels/first_try.h5')

## Second part: saving bottleneck features

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)

# build the VGG16 network
vggmodel = Sequential()
vggmodel.add(ZeroPadding2D((1, 1), input_shape=(3, img_width, img_height)))

vggmodel.add(Convolution2D(64, 3, 3, activation='relu', name='conv1_1'))
vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(64, 3, 3, activation='relu', name='conv1_2'))
vggmodel.add(MaxPooling2D((2, 2), strides=(2, 2)))

vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(128, 3, 3, activation='relu', name='conv2_1'))
vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(128, 3, 3, activation='relu', name='conv2_2'))
vggmodel.add(MaxPooling2D((2, 2), strides=(2, 2)))

vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(256, 3, 3, activation='relu', name='conv3_1'))
vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(256, 3, 3, activation='relu', name='conv3_2'))
vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(256, 3, 3, activation='relu', name='conv3_3'))
vggmodel.add(MaxPooling2D((2, 2), strides=(2, 2)))

vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(512, 3, 3, activation='relu', name='conv4_1'))
vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(512, 3, 3, activation='relu', name='conv4_2'))
vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(512, 3, 3, activation='relu', name='conv4_3'))
vggmodel.add(MaxPooling2D((2, 2), strides=(2, 2)))

vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(512, 3, 3, activation='relu', name='conv5_1'))
vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(512, 3, 3, activation='relu', name='conv5_2'))
vggmodel.add(ZeroPadding2D((1, 1)))
vggmodel.add(Convolution2D(512, 3, 3, activation='relu', name='conv5_3'))
vggmodel.add(MaxPooling2D((2, 2), strides=(2, 2)))

# load the weights of the VGG16 networks
# (trained on ImageNet, won the ILSVRC competition in 2014)
# note: when there is a complete match between your model definition
# and your weight savefile, you can simply call model.load_weights(filename)
assert os.path.exists(weights_path), 'Model weights not found (see "weights_path" variable in script).'
f = h5py.File(weights_path)
for k in range(f.attrs['nb_layers']):
    if k >= len(vggmodel.layers):
        # we don't look at the last (fully-connected) layers in the savefile
        break
    g = f['layer_{}'.format(k)]
    weights = [g['param_{}'.format(p)] for p in range(g.attrs['nb_params'])]
    vggmodel.layers[k].set_weights(weights)
f.close()
print('VGG16 Model loaded.')
print vggmodel.summary()

In [ ]:
if Train_Bottleneck:
    t_generator = datagen.flow_from_directory(
            train_data_dir,
            target_size=(img_width, img_height),
            batch_size=32,
            class_mode=None,
            shuffle=False)

    v_generator = datagen.flow_from_directory(
            validation_data_dir,
            target_size=(img_width, img_height),
            batch_size=32,
            class_mode=None,
            shuffle=False)

    bottleneck_features_train = vggmodel.predict_generator(t_generator, nb_train_samples)
    np.save(open(folder+'savedmodels/bottleneck_features_train.npy', 'w'), bottleneck_features_train)

    bottleneck_features_validation = vggmodel.predict_generator(v_generator, nb_validation_samples)
    np.save(open(folder+'savedmodels/bottleneck_features_validation.npy', 'w'), bottleneck_features_validation)

In [ ]:
train_data = np.load(open(folder+'savedmodels/bottleneck_features_train.npy'))
nb_train_samples = len(train_data)
train_labels = []
for i in range(nb_class):
    train_labels +=  list([i] * (nb_train_samples / nb_class))
    
validation_data = np.load(open(folder+'savedmodels/bottleneck_features_validation.npy'))
nb_validation_samples = len(validation_data)
validation_labels = []
for i in range(nb_class):
    validation_labels +=  list([i] * (nb_validation_samples / nb_class))

train_labels = np.array(train_labels)
validation_labels = np.array(validation_labels)
    
# print train_labels
# print validation_labels

print "Bottleneck features shape: Training set: ", train_data.shape, train_labels.shape
print "Bottleneck features shape: Validation set: ",validation_data.shape, validation_labels.shape

In [ ]:
topmodel = Sequential()
topmodel.add(Flatten(input_shape=train_data.shape[1:]))
topmodel.add(Dense(256, activation='relu'))
topmodel.add(Dropout(0.5))
topmodel.add(Dense(1, activation='sigmoid'))

topmodel.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
print topmodel.summary()
topmodel.fit(train_data, train_labels,
          nb_epoch=nb_epoch, batch_size=32,
          validation_data=(validation_data, validation_labels))
topmodel.save_weights(top_model_weights_path)

## Finally fine-tuning here

In [ ]:
# add the model on top of the convolutional base
# vggmodel = Sequential()
# model.add(vggmodel)
vggmodel.add(topmodel)
# set the first 25 layers (up to the last conv block)
# to non-trainable (weights will not be updated)
for layer in vggmodel.layers[:25]:
    layer.trainable = False


# compile the model with a SGD/momentum optimizer
# and a very slow learning rate.
vggmodel.compile(loss='binary_crossentropy',
              optimizer=optimizers.SGD(lr=1e-4, momentum=0.9),
              metrics=['accuracy'])
print vggmodel.summary()

In [ ]:
# prepare data augmentation configuration
train_datagen = ImageDataGenerator(
        rescale=1./255,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
        train_data_dir,
        target_size=(img_height, img_width),
        batch_size=32,
        class_mode='binary')

validation_generator = test_datagen.flow_from_directory(
        validation_data_dir,
        target_size=(img_height, img_width),
        batch_size=32,
        class_mode='binary')

In [ ]:
# fine-tune the model
model.fit_generator(
        train_generator,
        samples_per_epoch=nb_train_samples,
        nb_epoch=nb_epoch,
        validation_data=validation_generator,
        nb_val_samples=nb_validation_samples)